# File Integrity Checker with SHA256

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import os, hashlib, json

def hash_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def scan_directory(root):
    file_hashes = {}
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            full_path = os.path.join(dirpath, fname)
            try:
                file_hashes[full_path] = hash_file(full_path)
            except:
                pass
    return file_hashes

root_dir = "/content/sample_data"
baseline_file = "/content/baseline.json"
test_file = os.path.join(root_dir, "test_file.txt")

if not os.path.exists(test_file):
    with open(test_file, "w") as f:
        f.write("This is a test file to check integrity.\n")

if not os.path.exists(baseline_file):
    baseline = scan_directory(root_dir)
    with open(baseline_file, 'w') as f:
        json.dump(baseline, f)
    print("✅ Baseline created.")
else:
    with open(baseline_file) as f:
        baseline = json.load(f)
    current = scan_directory(root_dir)
    added = set(current) - set(baseline)
    removed = set(baseline) - set(current)
    modified = [f for f in current if f in baseline and current[f] != baseline[f]]
    print("➕ Added:", added)
    print("➖ Removed:", removed)
    print("✏️ Modified:", modified)
